# Create Complete `.tar` Archive for `intel_image_project`

Notebook này dùng để tạo lại file `.tar` **đầy đủ** cho project/dataset trên Google Drive, tránh lỗi archive cũ chỉ chứa thiếu file.

Notebook hỗ trợ 2 chế độ:

1. **DATA archive**: tạo `data_complete.tar`, chỉ chứa thư mục `data/`.
2. **PROJECT archive**: tạo `project_complete.tar`, chứa nhiều thư mục/file của project như `data/`, `notebooks/`, `models/`, `outputs/`, file `.json`, `.csv`, `.ipynb`, v.v.

Khuyến nghị cho train Colab: dùng `data_complete.tar` để copy nhanh dataset sang `/content`.


In [6]:
# =========================
# 1. MOUNT GOOGLE DRIVE
# =========================

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# =========================
# 2. IMPORTS
# =========================

from pathlib import Path
import os
import tarfile
import shutil
from collections import Counter


## 3. Tìm đúng thư mục project

Nếu notebook tìm thấy nhiều thư mục `intel_image_project`, hãy chọn đúng index ở biến `PROJECT_INDEX`.


In [8]:
# =========================
# 3. FIND PROJECT ROOT
# =========================

DRIVE_ROOT = Path("/content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project")

print("Using DRIVE_ROOT:")
print(DRIVE_ROOT)
print("Exists:", DRIVE_ROOT.exists())

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy DRIVE_ROOT: {DRIVE_ROOT}\n"
        "Hãy kiểm tra lại đường dẫn, đặc biệt là dấu cách trong tên thư mục."
    )

print("\nCác thư mục/file bên trong project:")
for path in sorted(DRIVE_ROOT.iterdir()):
    print("-", path.name)


Using DRIVE_ROOT:
/content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project
Exists: True

Các thư mục/file bên trong project:
- data
- models
- outputs


## 4. Kiểm tra cấu trúc project trước khi tạo `.tar`

Cell dưới sẽ kiểm tra các thành phần quan trọng mà notebook train thường cần:

- `data/raw`
- `data/splits/train.csv`
- `data/splits/val.csv`
- `data/splits/test.csv`
- `data/metadata/class_to_idx.json`


In [9]:
# =========================
# 4. VALIDATE PROJECT STRUCTURE
# =========================

DATA_DIR = DRIVE_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
SPLIT_DIR = DATA_DIR / "splits"
METADATA_DIR = DATA_DIR / "metadata"

required_paths = [
    DATA_DIR,
    RAW_DIR,
    SPLIT_DIR,
    SPLIT_DIR / "train.csv",
    SPLIT_DIR / "val.csv",
    SPLIT_DIR / "test.csv",
    METADATA_DIR,
    METADATA_DIR / "class_to_idx.json",
]

print("Required path check:")
missing = []

for path in required_paths:
    ok = path.exists()
    print(f"{'OK   ' if ok else 'MISS '} {path}")
    if not ok:
        missing.append(path)

print("\nSummary:")
print("Missing count:", len(missing))

if missing:
    print("\nMissing paths:")
    for path in missing:
        print("-", path)
else:
    print("Tất cả file/thư mục quan trọng đều tồn tại.")


Required path check:
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/raw
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/splits
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/splits/train.csv
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/splits/val.csv
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/splits/test.csv
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/metadata
OK    /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data/metadata/class_to_idx.json

Summary:
Missing count: 0
Tất cả file/thư mục quan trọng đều tồn tại.


In [10]:
# =========================
# 5. COUNT FILES IN ORIGINAL DATA
# =========================

def count_files_by_suffix(root: Path):
    suffix_counter = Counter()
    total_files = 0

    if not root.exists():
        return total_files, suffix_counter

    for path in root.rglob("*"):
        if path.is_file():
            total_files += 1
            suffix = path.suffix.lower() if path.suffix else "<no_ext>"
            suffix_counter[suffix] += 1

    return total_files, suffix_counter


total_files, suffix_counter = count_files_by_suffix(DATA_DIR)

print("DATA_DIR:", DATA_DIR)
print("Total files in data/:", total_files)
print("\nTop file suffixes:")
for suffix, count in suffix_counter.most_common(20):
    print(f"{suffix}: {count}")

image_count = sum(
    suffix_counter.get(ext, 0)
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
)

csv_count = suffix_counter.get(".csv", 0)
json_count = suffix_counter.get(".json", 0)

print("\nImage files:", image_count)
print("CSV files:", csv_count)
print("JSON files:", json_count)


DATA_DIR: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data
Total files in data/: 24351

Top file suffixes:
.jpg: 24338
.csv: 8
.txt: 2
.json: 2
.md: 1

Image files: 24338
CSV files: 8
JSON files: 2


## 6. Tạo `data_complete.tar`

Archive này chỉ chứa `data/`. Đây là lựa chọn khuyên dùng để train nhanh trên Colab.

File tạo ra:

```text
<intel_image_project>/data_complete.tar
```


In [11]:
# =========================
# 6. CREATE DATA ARCHIVE
# =========================

DATA_TAR_PATH = DRIVE_ROOT / "data_complete.tar"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy DATA_DIR: {DATA_DIR}")

if DATA_TAR_PATH.exists():
    print("Removing old archive:", DATA_TAR_PATH)
    DATA_TAR_PATH.unlink()

print("Creating:", DATA_TAR_PATH)

with tarfile.open(DATA_TAR_PATH, "w") as tar:
    # arcname="data" để khi giải nén sẽ ra thư mục data/
    tar.add(DATA_DIR, arcname="data")

print("Done.")
print("Archive path:", DATA_TAR_PATH)
print("Archive size MB:", DATA_TAR_PATH.stat().st_size / 1024 / 1024)


Creating: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data_complete.tar
Done.
Archive path: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data_complete.tar
Archive size MB: 399.228515625


## 7. Kiểm tra nội dung `data_complete.tar`

Cell này kiểm tra archive vừa tạo có đủ ảnh, CSV, JSON và các file quan trọng không.


In [12]:
# =========================
# 7. INSPECT DATA ARCHIVE
# =========================

def inspect_tar(tar_path: Path, max_preview: int = 50):
    if not tar_path.exists():
        raise FileNotFoundError(tar_path)

    with tarfile.open(tar_path, "r") as tar:
        members = tar.getmembers()

    names = [m.name for m in members]
    file_names = [m.name for m in members if m.isfile()]

    print("Archive:", tar_path)
    print("Size MB:", tar_path.stat().st_size / 1024 / 1024)
    print("Total entries:", len(names))
    print("Total files:", len(file_names))

    suffix_counter = Counter(
        Path(name).suffix.lower() if Path(name).suffix else "<no_ext>"
        for name in file_names
    )

    print("\nTop file suffixes:")
    for suffix, count in suffix_counter.most_common(20):
        print(f"{suffix}: {count}")

    image_count = sum(
        suffix_counter.get(ext, 0)
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    )

    print("\nImage files:", image_count)
    print("CSV files:", suffix_counter.get(".csv", 0))
    print("JSON files:", suffix_counter.get(".json", 0))

    important_files = [
        "data/splits/train.csv",
        "data/splits/val.csv",
        "data/splits/test.csv",
        "data/metadata/class_to_idx.json",
    ]

    print("\nImportant files:")
    for file in important_files:
        print(f"{file}: {file in names}")

    print("\nFirst entries:")
    for name in names[:max_preview]:
        print(name)

    return names


data_tar_names = inspect_tar(DATA_TAR_PATH)


Archive: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/data_complete.tar
Size MB: 399.228515625
Total entries: 24370
Total files: 24351

Top file suffixes:
.jpg: 24338
.csv: 8
.txt: 2
.json: 2
.md: 1

Image files: 24338
CSV files: 8
JSON files: 2

Important files:
data/splits/train.csv: True
data/splits/val.csv: True
data/splits/test.csv: True
data/metadata/class_to_idx.json: True

First entries:
data
data/metadata
data/metadata/class_names.txt
data/metadata/class_to_idx.json
data/metadata/clean_metadata_all.csv
data/metadata/dataset_notes.md
data/metadata/duplicates_cross_split.csv
data/metadata/duplicates_within_split.csv
data/metadata/preprocessing_summary.json
data/metadata/raw_metadata_all.csv
data/metadata/source.txt
data/raw
data/raw/seg_pred
data/raw/seg_pred/10004.jpg
data/raw/seg_pred/10005.jpg
data/raw/seg_pred/10012.jpg
data/raw/seg_pred/10013.jpg
data/raw/seg_pred/10017.jpg
data/raw/seg_pred/10021.jpg
data/raw/seg_pred/1003.jpg
dat

## 8. Tạo `project_complete.tar` nếu cần

Archive này chứa nhiều thành phần hơn `data/`, phù hợp nếu bạn muốn backup toàn bộ project.

Mặc định sẽ **không đưa các file `.tar`, `.zip`, `.7z`, `.rar` cũ vào archive** để tránh archive lồng archive làm file quá lớn.


In [16]:
# =========================
# 8. CREATE FULL PROJECT ARCHIVE
# =========================

CREATE_PROJECT_TAR = True

PROJECT_TAR_PATH = DRIVE_ROOT / "project_complete.tar"

# Những thư mục/file lớn hoặc không cần thiết có thể loại trừ.
EXCLUDE_SUFFIXES = {
    ".tar",
    ".zip",
    ".7z",
    ".rar",
}

EXCLUDE_DIR_NAMES = {
    "__pycache__",
    ".ipynb_checkpoints",
}

def should_exclude(path: Path) -> bool:
    # Không đưa chính archive vào archive.
    if path.resolve() in {DATA_TAR_PATH.resolve(), PROJECT_TAR_PATH.resolve()}:
        return True

    if path.name in EXCLUDE_DIR_NAMES:
        return True

    if path.is_file() and path.suffix.lower() in EXCLUDE_SUFFIXES:
        return True

    return False


if CREATE_PROJECT_TAR:
    if PROJECT_TAR_PATH.exists():
        print("Removing old archive:", PROJECT_TAR_PATH)
        PROJECT_TAR_PATH.unlink()

    print("Creating:", PROJECT_TAR_PATH)

    with tarfile.open(PROJECT_TAR_PATH, "w") as tar:
        for item in DRIVE_ROOT.rglob("*"):
            if should_exclude(item):
                continue

            # Bỏ qua file/thư mục nằm trong thư mục bị exclude.
            if any(part in EXCLUDE_DIR_NAMES for part in item.parts):
                continue

            arcname = item.relative_to(DRIVE_ROOT.parent)
            tar.add(item, arcname=str(arcname))

    print("Done.")
    print("Archive path:", PROJECT_TAR_PATH)
    print("Archive size MB:", PROJECT_TAR_PATH.stat().st_size / 1024 / 1024)
else:
    print("CREATE_PROJECT_TAR = False, bỏ qua bước tạo project_complete.tar.")


Creating: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/project_complete.tar
Done.
Archive path: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/project_complete.tar
Archive size MB: 3282.9296875


## 9. Kiểm tra `project_complete.tar`

Chạy cell này nếu bạn đã bật `CREATE_PROJECT_TAR = True`.


In [17]:
# =========================
# 9. INSPECT PROJECT ARCHIVE
# =========================

if CREATE_PROJECT_TAR:
    project_tar_names = inspect_tar(PROJECT_TAR_PATH)
else:
    print("Không có project_complete.tar để kiểm tra.")


Archive: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/project_complete.tar
Size MB: 3282.9296875
Total entries: 114770
Total files: 114674

Top file suffixes:
.jpg: 114389
.png: 94
.csv: 87
.npz: 56
.json: 18
.txt: 9
.joblib: 9
.pth: 9
.md: 3

Image files: 114483
CSV files: 87
JSON files: 18

Important files:
data/splits/train.csv: False
data/splits/val.csv: False
data/splits/test.csv: False
data/metadata/class_to_idx.json: False

First entries:
intel_image_project/data
intel_image_project/data/metadata
intel_image_project/data/metadata/class_names.txt
intel_image_project/data/metadata/class_to_idx.json
intel_image_project/data/metadata/clean_metadata_all.csv
intel_image_project/data/metadata/dataset_notes.md
intel_image_project/data/metadata/duplicates_cross_split.csv
intel_image_project/data/metadata/duplicates_within_split.csv
intel_image_project/data/metadata/preprocessing_summary.json
intel_image_project/data/metadata/raw_metadata_all.csv

## 10. Copy nhanh `data_complete.tar` sang `/content` và giải nén thử

Cell này giúp bạn kiểm tra archive có giải nén đúng không.

Sau khi giải nén, dữ liệu sẽ nằm tại:

```text
/content/intel_image_project/data
```


In [14]:
# =========================
# 10. TEST EXTRACT DATA ARCHIVE TO /content
# =========================

LOCAL_ROOT = Path("/content/intel_image_project")
LOCAL_TAR_PATH = Path("/content/data_complete.tar")

# Xóa bản test cũ nếu có.
if LOCAL_ROOT.exists():
    print("Removing old LOCAL_ROOT:", LOCAL_ROOT)
    shutil.rmtree(LOCAL_ROOT)

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Copying archive to local /content...")
shutil.copy2(DATA_TAR_PATH, LOCAL_TAR_PATH)

print("Extracting archive...")
with tarfile.open(LOCAL_TAR_PATH, "r") as tar:
    tar.extractall(path=LOCAL_ROOT)

print("Done.")
print("LOCAL_ROOT:", LOCAL_ROOT)
print("LOCAL data exists:", (LOCAL_ROOT / "data").exists())

print("\nLocal data tree:")
for path in sorted((LOCAL_ROOT / "data").glob("*")):
    print(path)


Copying archive to local /content...
Extracting archive...


/tmp/ipykernel_19075/144922421.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=LOCAL_ROOT)


Done.
LOCAL_ROOT: /content/intel_image_project
LOCAL data exists: True

Local data tree:
/content/intel_image_project/data/metadata
/content/intel_image_project/data/raw
/content/intel_image_project/data/splits


In [15]:
# =========================
# 11. VERIFY EXTRACTED LOCAL DATA
# =========================

LOCAL_DATA_DIR = LOCAL_ROOT / "data"

required_local_paths = [
    LOCAL_DATA_DIR,
    LOCAL_DATA_DIR / "raw",
    LOCAL_DATA_DIR / "splits" / "train.csv",
    LOCAL_DATA_DIR / "splits" / "val.csv",
    LOCAL_DATA_DIR / "splits" / "test.csv",
    LOCAL_DATA_DIR / "metadata" / "class_to_idx.json",
]

print("Local required path check:")
for path in required_local_paths:
    print(f"{'OK   ' if path.exists() else 'MISS '} {path}")

total_local_files, local_suffix_counter = count_files_by_suffix(LOCAL_DATA_DIR)

print("\nTotal local files:", total_local_files)
print("Local image files:", sum(local_suffix_counter.get(ext, 0) for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]))
print("Local CSV files:", local_suffix_counter.get(".csv", 0))
print("Local JSON files:", local_suffix_counter.get(".json", 0))


Local required path check:
OK    /content/intel_image_project/data
OK    /content/intel_image_project/data/raw
OK    /content/intel_image_project/data/splits/train.csv
OK    /content/intel_image_project/data/splits/val.csv
OK    /content/intel_image_project/data/splits/test.csv
OK    /content/intel_image_project/data/metadata/class_to_idx.json

Total local files: 24351
Local image files: 24338
Local CSV files: 8
Local JSON files: 2


## 12. Cách dùng archive này trong notebook train

Trong notebook train, bạn có thể dùng logic sau:

```python
DRIVE_ROOT = Path("/content/drive/MyDrive/.../intel_image_project")
LOCAL_ROOT = Path("/content/intel_image_project")

shutil.copy2(DRIVE_ROOT / "data_complete.tar", "/content/data_complete.tar")

with tarfile.open("/content/data_complete.tar", "r") as tar:
    tar.extractall(path=LOCAL_ROOT)

PROJECT_ROOT = LOCAL_ROOT
```

Sau đó chạy lại các cell tạo `DATA_DIR`, `Dataset`, `DataLoader`.
